In [3]:
import pandas as pd
df2 = pd.read_csv('Social_Media_Advertising.csv')
print(df2.shape)
print(df2.info())
print(df2.isnull().sum())

(300000, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Campaign_ID       300000 non-null  int64  
 1   Target_Audience   300000 non-null  object 
 2   Campaign_Goal     300000 non-null  object 
 3   Duration          300000 non-null  object 
 4   Channel_Used      300000 non-null  object 
 5   Conversion_Rate   300000 non-null  float64
 6   Acquisition_Cost  300000 non-null  object 
 7   ROI               300000 non-null  float64
 8   Location          300000 non-null  object 
 9   Language          300000 non-null  object 
 10  Clicks            300000 non-null  int64  
 11  Impressions       300000 non-null  int64  
 12  Engagement_Score  300000 non-null  int64  
 13  Customer_Segment  300000 non-null  object 
 14  Date              300000 non-null  object 
 15  Company           300000 non-null  object 
dtypes: floa

In [9]:
# Check Sample_Data
print(Sample_Data.shape)
print(Sample_Data['Channel_Used'].value_counts())
# Save sample data to CSV file
Sample_Data.to_csv('social_media_sample.csv', index=False)
print("Sample data saved successfully")

(2000, 16)
Channel_Used
Facebook     530
Pinterest    508
Twitter      481
Instagram    481
Name: count, dtype: int64
Sample data saved successfully


In [17]:
# Step 1: Drop unnecessary columns
columns_to_drop = ['Campaign_ID', 'Date', 'Company', 'Location', 'Language']
all_clean = Sample_Data.drop(columns=columns_to_drop)

# Step 2: Fix Acquisition_Cost
all_clean['Acquisition_Cost'] = all_clean['Acquisition_Cost'].str.replace('$', '').str.replace(',', '').astype(float)

# Step 3: Fix Duration
all_clean['Duration'] = all_clean['Duration'].str.replace(' Days', '').astype(int)

# Verify
print(all_clean.shape)
print(all_clean.columns.tolist())
print(all_clean['Channel_Used'].value_counts())
print(all_clean.isnull().sum())

(2000, 11)
['Target_Audience', 'Campaign_Goal', 'Duration', 'Channel_Used', 'Conversion_Rate', 'Acquisition_Cost', 'ROI', 'Clicks', 'Impressions', 'Engagement_Score', 'Customer_Segment']
Channel_Used
Facebook     530
Pinterest    508
Twitter      481
Instagram    481
Name: count, dtype: int64
Target_Audience     0
Campaign_Goal       0
Duration            0
Channel_Used        0
Conversion_Rate     0
Acquisition_Cost    0
ROI                 0
Clicks              0
Impressions         0
Engagement_Score    0
Customer_Segment    0
dtype: int64


In [21]:
# Step 4: Encode categorical columns
all_encoded = pd.get_dummies(all_clean, 
                              columns=['Target_Audience', 
                                       'Campaign_Goal',
                                       'Channel_Used',
                                       'Customer_Segment'])

print(all_encoded.shape)

# Step 5: Create performance score
all_encoded['performance_score'] = (
    (all_encoded['ROI'] / all_encoded['ROI'].max()) +
    (all_encoded['Conversion_Rate'] / all_encoded['Conversion_Rate'].max()) +
    (all_encoded['Engagement_Score'] / all_encoded['Engagement_Score'].max()) +
    (1 - all_encoded['Acquisition_Cost'] / all_encoded['Acquisition_Cost'].max()) +
    (all_encoded['Clicks'] / all_encoded['Clicks'].max()) +
    (all_encoded['Impressions'] / all_encoded['Impressions'].max())
) / 6

# Step 6: Create target variable
threshold = all_encoded['performance_score'].mean()
all_encoded['campaign_success'] = (all_encoded['performance_score'] > threshold).astype(int)

print(f"Threshold: {threshold:.4f}")
print(all_encoded['campaign_success'].value_counts())
print(all_encoded['campaign_success'].value_counts(normalize=True) * 100)

(2000, 29)
Threshold: 0.4626
campaign_success
1    1036
0     964
Name: count, dtype: int64
campaign_success
1    51.8
0    48.2
Name: proportion, dtype: float64


In [22]:
# Step 7: Define X and y
y = all_encoded['campaign_success']

X = all_encoded.drop(columns=['campaign_success', 
                                'performance_score',
                                'ROI',
                                'Conversion_Rate',
                                'Engagement_Score',
                                'Clicks'])

print(X.shape)
print(X.columns.tolist())
print(y.value_counts())

(2000, 25)
['Duration', 'Acquisition_Cost', 'Impressions', 'Target_Audience_All Ages', 'Target_Audience_Men 18-24', 'Target_Audience_Men 25-34', 'Target_Audience_Men 35-44', 'Target_Audience_Men 45-60', 'Target_Audience_Women 18-24', 'Target_Audience_Women 25-34', 'Target_Audience_Women 35-44', 'Target_Audience_Women 45-60', 'Campaign_Goal_Brand Awareness', 'Campaign_Goal_Increase Sales', 'Campaign_Goal_Market Expansion', 'Campaign_Goal_Product Launch', 'Channel_Used_Facebook', 'Channel_Used_Instagram', 'Channel_Used_Pinterest', 'Channel_Used_Twitter', 'Customer_Segment_Fashion', 'Customer_Segment_Food', 'Customer_Segment_Health', 'Customer_Segment_Home', 'Customer_Segment_Technology']
campaign_success
1    1036
0     964
Name: count, dtype: int64


In [23]:
# Step 8: Split data
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")
print(y_train.value_counts())
print(y_test.value_counts())

# Step 9: Build models
# Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=5000)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

# Decision Tree
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# Results
print("\nLogistic Regression:")
print(f"Accuracy: {accuracy_score(y_test, lr_pred):.4f}")
print(classification_report(y_test, lr_pred))

print("\nDecision Tree:")
print(f"Accuracy: {accuracy_score(y_test, dt_pred):.4f}")
print(classification_report(y_test, dt_pred))

print("\nRandom Forest:")
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(classification_report(y_test, rf_pred))

Training rows: 1600
Testing rows: 400
campaign_success
1    836
0    764
Name: count, dtype: int64
campaign_success
1    200
0    200
Name: count, dtype: int64

Logistic Regression:
Accuracy: 0.7950
              precision    recall  f1-score   support

           0       0.83      0.74      0.78       200
           1       0.77      0.85      0.81       200

    accuracy                           0.80       400
   macro avg       0.80      0.79      0.79       400
weighted avg       0.80      0.80      0.79       400


Decision Tree:
Accuracy: 0.7850
              precision    recall  f1-score   support

           0       0.78      0.80      0.79       200
           1       0.79      0.78      0.78       200

    accuracy                           0.79       400
   macro avg       0.79      0.79      0.78       400
weighted avg       0.79      0.79      0.78       400


Random Forest:
Accuracy: 0.7450
              precision    recall  f1-score   support

           0       0.78   

In [24]:
print(f"LR Training accuracy: {accuracy_score(y_train, lr.predict(X_train_scaled)):.4f}")
print(f"LR Testing accuracy: {accuracy_score(y_test, lr_pred):.4f}")

print(f"DT Training accuracy: {accuracy_score(y_train, dt.predict(X_train)):.4f}")
print(f"DT Testing accuracy: {accuracy_score(y_test, dt_pred):.4f}")

LR Training accuracy: 0.7919
LR Testing accuracy: 0.7950
DT Training accuracy: 0.7925
DT Testing accuracy: 0.7850


In [25]:
import pickle

# Save Logistic Regression as final model
with open('campaign_model.pkl', 'wb') as f:
    pickle.dump(lr, f)

# Save scaler too as we need it for demo
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model and scaler saved successfully")

Model and scaler saved successfully


In [1]:
!pip install gradio

   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
   - -------------------------------------- 1.3/30.7 MB 7.1 MB/s eta 0:00:05
   ---- ----------------------------------- 3.1/30.7 MB 8.1 MB/s eta 0:00:04
   ------- -------------------------------- 5.8/30.7 MB 9.6 MB/s eta 0:00:03
   ----------- ---------------------------- 9.2/30.7 MB 11.3 MB/s eta 0:00:02
   ----------------- ---------------------- 13.1/30.7 MB 12.9 MB/s eta 0:00:02
   ---------------------- ----------------- 17.3/30.7 MB 14.3 MB/s eta 0:00:01
   ----------------------------- ---------- 22.3/30.7 MB 15.7 MB/s eta 0:00:01
   ----------------------------------- ---- 27.0/30.7 MB 16.6 MB/s eta 0:00:01
   ---------------------------------------  30.7/30.7 MB 16.9 MB/s eta 0:00:01
   ---------------------------------------- 30.7/30.7 MB 15.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/784.9 kB ? eta -:--:--
   --------------------------------------- 784.9/784.9 kB 14.3 MB/s eta 0:00

In [1]:
import gradio as gr
import pickle
import numpy as np

# Load model and scaler
with open('campaign_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

def predict_campaign(duration, acquisition_cost, impressions,
                     target_audience, campaign_goal, 
                     channel, customer_segment):
    
    # Create input array
    input_dict = {
        'Duration': duration,
        'Acquisition_Cost': acquisition_cost,
        'Impressions': impressions,
        'Target_Audience_All Ages': 1 if target_audience == 'All Ages' else 0,
        'Target_Audience_Men 18-24': 1 if target_audience == 'Men 18-24' else 0,
        'Target_Audience_Men 25-34': 1 if target_audience == 'Men 25-34' else 0,
        'Target_Audience_Men 35-44': 1 if target_audience == 'Men 35-44' else 0,
        'Target_Audience_Men 45-60': 1 if target_audience == 'Men 45-60' else 0,
        'Target_Audience_Women 18-24': 1 if target_audience == 'Women 18-24' else 0,
        'Target_Audience_Women 25-34': 1 if target_audience == 'Women 25-34' else 0,
        'Target_Audience_Women 35-44': 1 if target_audience == 'Women 35-44' else 0,
        'Target_Audience_Women 45-60': 1 if target_audience == 'Women 45-60' else 0,
        'Campaign_Goal_Brand Awareness': 1 if campaign_goal == 'Brand Awareness' else 0,
        'Campaign_Goal_Increase Sales': 1 if campaign_goal == 'Increase Sales' else 0,
        'Campaign_Goal_Market Expansion': 1 if campaign_goal == 'Market Expansion' else 0,
        'Campaign_Goal_Product Launch': 1 if campaign_goal == 'Product Launch' else 0,
        'Channel_Used_Facebook': 1 if channel == 'Facebook' else 0,
        'Channel_Used_Instagram': 1 if channel == 'Instagram' else 0,
        'Channel_Used_Pinterest': 1 if channel == 'Pinterest' else 0,
        'Channel_Used_Twitter': 1 if channel == 'Twitter' else 0,
        'Customer_Segment_Fashion': 1 if customer_segment == 'Fashion' else 0,
        'Customer_Segment_Food': 1 if customer_segment == 'Food' else 0,
        'Customer_Segment_Health': 1 if customer_segment == 'Health' else 0,
        'Customer_Segment_Home': 1 if customer_segment == 'Home' else 0,
        'Customer_Segment_Technology': 1 if customer_segment == 'Technology' else 0,
    }
    
    # Convert to array and scale
    input_array = np.array(list(input_dict.values())).reshape(1, -1)
    input_scaled = scaler.transform(input_array)
    
    # Predict
    prediction = model.predict(input_scaled)[0]
    probability = model.predict_proba(input_scaled)[0]
    
    if prediction == 1:
        return f"YES - Campaign likely to SUCCEED. Confidence: {probability[1]*100:.1f}%"
    else:
        return f"NO - Campaign likely to UNDERPERFORM. Confidence: {probability[0]*100:.1f}%"

# Create interface
demo = gr.Interface(
    fn=predict_campaign,
    inputs=[
        gr.Slider(minimum=15, maximum=60, step=15, label="Campaign Duration (Days)"),
        gr.Slider(minimum=500, maximum=15000, step=500, label="Campaign Budget (USD)"),
        gr.Slider(minimum=1000, maximum=120000, step=1000, label="Expected Impressions"),
        gr.Dropdown(['All Ages', 'Men 18-24', 'Men 25-34', 'Men 35-44', 'Men 45-60',
                     'Women 18-24', 'Women 25-34', 'Women 35-44', 'Women 45-60'], 
                    label="Target Audience"),
        gr.Dropdown(['Brand Awareness', 'Increase Sales', 
                     'Market Expansion', 'Product Launch'], 
                    label="Campaign Goal"),
        gr.Dropdown(['Facebook', 'Instagram', 'Pinterest', 'Twitter'], 
                    label="Channel"),
        gr.Dropdown(['Fashion', 'Food', 'Health', 'Home', 'Technology'], 
                    label="Customer Segment"),
    ],
    outputs=gr.Textbox(label="Prediction"),
    title="Marketing Campaign Performance Predictor",
    description="Enter your campaign details to predict whether it will succeed or not."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [1]:
# Simple test
import numpy as np

# Test input 1: Low budget short campaign
test1 = np.array([15, 500, 1000, 0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0]).reshape(1,-1)
test1_scaled = scaler.transform(test1)
pred1 = model.predict(test1_scaled)[0]
prob1 = model.predict_proba(test1_scaled)[0]
print(f"Test 1: {'YES' if pred1==1 else 'NO'} - Prob: {prob1}")

# Test input 2: High budget long campaign
test2 = np.array([60, 15000, 120000, 0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0]).reshape(1,-1)
test2_scaled = scaler.transform(test2)
pred2 = model.predict(test2_scaled)[0]
prob2 = model.predict_proba(test2_scaled)[0]
print(f"Test 2: {'YES' if pred2==1 else 'NO'} - Prob: {prob2}")

NameError: name 'scaler' is not defined

In [5]:
import gradio as gr
import pickle
import numpy as np

# Load model and scaler
with open('campaign_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

def predict_campaign(duration, acquisition_cost, impressions,
                     target_audience, campaign_goal,
                     channel, customer_segment):

    input_dict = {
        'Duration': duration,
        'Acquisition_Cost': acquisition_cost,
        'Impressions': impressions,
        'Target_Audience_All Ages': 1 if target_audience == 'All Ages' else 0,
        'Target_Audience_Men 18-24': 1 if target_audience == 'Men 18-24' else 0,
        'Target_Audience_Men 25-34': 1 if target_audience == 'Men 25-34' else 0,
        'Target_Audience_Men 35-44': 1 if target_audience == 'Men 35-44' else 0,
        'Target_Audience_Men 45-60': 1 if target_audience == 'Men 45-60' else 0,
        'Target_Audience_Women 18-24': 1 if target_audience == 'Women 18-24' else 0,
        'Target_Audience_Women 25-34': 1 if target_audience == 'Women 25-34' else 0,
        'Target_Audience_Women 35-44': 1 if target_audience == 'Women 35-44' else 0,
        'Target_Audience_Women 45-60': 1 if target_audience == 'Women 45-60' else 0,
        'Campaign_Goal_Brand Awareness': 1 if campaign_goal == 'Brand Awareness' else 0,
        'Campaign_Goal_Increase Sales': 1 if campaign_goal == 'Increase Sales' else 0,
        'Campaign_Goal_Market Expansion': 1 if campaign_goal == 'Market Expansion' else 0,
        'Campaign_Goal_Product Launch': 1 if campaign_goal == 'Product Launch' else 0,
        'Channel_Used_Facebook': 1 if channel == 'Facebook' else 0,
        'Channel_Used_Instagram': 1 if channel == 'Instagram' else 0,
        'Channel_Used_Pinterest': 1 if channel == 'Pinterest' else 0,
        'Channel_Used_Twitter': 1 if channel == 'Twitter' else 0,
        'Customer_Segment_Fashion': 1 if customer_segment == 'Fashion' else 0,
        'Customer_Segment_Food': 1 if customer_segment == 'Food' else 0,
        'Customer_Segment_Health': 1 if customer_segment == 'Health' else 0,
        'Customer_Segment_Home': 1 if customer_segment == 'Home' else 0,
        'Customer_Segment_Technology': 1 if customer_segment == 'Technology' else 0,
    }

    input_array = np.array(list(input_dict.values())).reshape(1, -1)
    input_scaled = scaler.transform(input_array)
    prediction = model.predict(input_scaled)[0]
    probability = model.predict_proba(input_scaled)[0]

    if prediction == 1:
        return (
            f"✅ YES - This campaign is likely to SUCCEED\n\n"
            f"Confidence Score: {probability[1]*100:.1f}%\n\n"
            f"Based on the selected targeting, channel, goal and budget "
            f"this campaign shows strong potential for success."
        )
    else:
        return (
            f"❌ NO - This campaign is likely to UNDERPERFORM\n\n"
            f"Confidence Score: {probability[0]*100:.1f}%\n\n"
            f"Consider increasing your budget, extending campaign duration "
            f"or adjusting your target audience for better results."
        )

demo = gr.Interface(
    fn=predict_campaign,
    inputs=[
        gr.Slider(minimum=15, maximum=60, step=15,
                  label="Campaign Duration (Days)", value=30),
        gr.Slider(minimum=500, maximum=15000, step=500,
                  label="Campaign Budget (USD)", value=7000),
        gr.Slider(minimum=1000, maximum=120000, step=1000,
                  label="Expected Impressions", value=50000),
        gr.Dropdown(
            ['All Ages', 'Men 18-24', 'Men 25-34', 'Men 35-44', 'Men 45-60',
             'Women 18-24', 'Women 25-34', 'Women 35-44', 'Women 45-60'],
            label="Target Audience", value="Women 25-34"),
        gr.Dropdown(
            ['Brand Awareness', 'Increase Sales',
             'Market Expansion', 'Product Launch'],
            label="Campaign Goal", value="Increase Sales"),
        gr.Dropdown(
            ['Facebook', 'Instagram', 'Pinterest', 'Twitter'],
            label="Channel", value="Instagram"),
        gr.Dropdown(
            ['Fashion', 'Food', 'Health', 'Home', 'Technology'],
            label="Customer Segment", value="Fashion"),
    ],
    outputs=gr.Textbox(label="Prediction Result", lines=6),
    title="aeoflo | Marketing Campaign Performance Predictor",
    description="""
    ### Powered by aeoflo AB | Built by Akash Chaudhary | MSc Data Science for Business | University of Stirling
    
    This tool predicts whether your marketing campaign will succeed or underperform 
    using machine learning trained on 2000 real campaigns across Facebook, Instagram, 
    Twitter and Pinterest achieving 79.5% accuracy.
    
    Fill in your campaign details below and click Submit to get your prediction.
    """,
    theme=gr.themes.Soft(),
    examples=[
        [60, 15000, 120000, "Women 35-44", "Increase Sales", "Instagram", "Fashion"],
        [15, 500, 1000, "Men 18-24", "Brand Awareness", "Twitter", "Technology"],
        [30, 7000, 50000, "All Ages", "Product Launch", "Facebook", "Health"],
    ]
)

demo.launch()

C:\Users\Aakash chaudhary\anaconda3\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [10]:
import os
print(os.getcwd())

C:\Users\Aakash chaudhary
